## Plant disease detection

Different DL Models
* LeNet-5
* AlexNet
* VGG16
* ResNet
* denseNet
* mobileNet
* EfficientNet
* Vision Transformer


## Download the dataset

In [1]:
import kagglehub
path = kagglehub.dataset_download("tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh")

100%|██████████| 270M/270M [00:07<00:00, 36.4MB/s]

Extracting files...


In [2]:
path

'/root/.cache/kagglehub/datasets/tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh/versions/1'

## Add all imports

In [3]:
import os
import torch
import torchvision
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as func
import matplotlib.pyplot as plt
import albumentations
import torch.nn as nn
from sklearn import metrics , model_selection
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision import transforms
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from torchvision import transforms
from torch.utils.data import Dataset


## image organising and processing

In [4]:
os.listdir(path)

['Pumpkin Leaf Diseases Dataset From Bangladesh']

In [5]:
data = []
dataset_path = os.path.join(path, "Pumpkin Leaf Diseases Dataset From Bangladesh/Original Dataset")
for label in os.listdir(dataset_path):
  class_path = os.path.join(dataset_path , label)
  if os.path.isdir(class_path):
    for img in os.listdir(class_path):
      img_path = os.path.join(class_path , img)
      data.append([img_path , label])

In [6]:
data[:5]

[['/root/.cache/kagglehub/datasets/tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh/versions/1/Pumpkin Leaf Diseases Dataset From Bangladesh/Original Dataset/Healthy Leaf/Healthy Leaf (73).jpg',
  'Healthy Leaf'],
 ['/root/.cache/kagglehub/datasets/tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh/versions/1/Pumpkin Leaf Diseases Dataset From Bangladesh/Original Dataset/Healthy Leaf/Healthy Leaf (277).jpg',
  'Healthy Leaf'],
 ['/root/.cache/kagglehub/datasets/tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh/versions/1/Pumpkin Leaf Diseases Dataset From Bangladesh/Original Dataset/Healthy Leaf/Healthy Leaf (185).jpg',
  'Healthy Leaf'],
 ['/root/.cache/kagglehub/datasets/tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh/versions/1/Pumpkin Leaf Diseases Dataset From Bangladesh/Original Dataset/Healthy Leaf/Healthy Leaf (9).jpg',
  'Healthy Leaf'],
 ['/root/.cache/kagglehub/datasets/tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh/versions/1/Pumpkin Leaf 

In [7]:
dfx = pd.DataFrame(data , columns=['img_path' , 'label'])
dfx.sample(5)

,img_path,label
242,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Healthy Leaf
1065,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Powdery_Mildew
150,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Healthy Leaf
1872,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Mosaic Disease
1762,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Mosaic Disease


In [ ]:
os

In [8]:
dfx['label'].value_counts()

,count
label,
Healthy Leaf,400
Bacterial Leaf Spot,400
Powdery_Mildew,400
Downy Mildew,400
Mosaic Disease,400


In [9]:
train_dfx , val_dfx = train_test_split(dfx ,
                                       test_size=0.2,
                                       stratify = dfx['label'],
                                       random_state = 42
                                       )

In [10]:
print(f"labels value count in train {train_dfx['label'].value_counts()}\n labels value count in test : {val_dfx['label'].value_counts()}")

labels value count in train label
Downy Mildew           320
Healthy Leaf           320
Bacterial Leaf Spot    320
Mosaic Disease         320
Powdery_Mildew         320
Name: count, dtype: int64
 labels value count in test : label
Powdery_Mildew         80
Mosaic Disease         80
Downy Mildew           80
Bacterial Leaf Spot    80
Healthy Leaf           80
Name: count, dtype: int64


In [11]:
train_dfx.reset_index(drop = True)
val_dfx.reset_index(drop = True)

,img_path,label
0,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Powdery_Mildew
1,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Mosaic Disease
2,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Powdery_Mildew
3,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Downy Mildew
4,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Bacterial Leaf Spot
...,...,...
395,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Downy Mildew
396,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Downy Mildew
397,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Downy Mildew
398,/root/.cache/kagglehub/datasets/tahmidmir/pump...,Mosaic Disease


In [12]:
le = LabelEncoder()
train_dfx['label_encoded'] = le.fit_transform(train_dfx['label'])
val_dfx['label_encoded'] = le.fit_transform(val_dfx['label'])

In [13]:
print(train_dfx[ ['label' , 'label_encoded']].sample(5))

                    label  label_encoded
1347         Downy Mildew              1
510   Bacterial Leaf Spot              0
1900       Mosaic Disease              3
1501         Downy Mildew              1
834        Powdery_Mildew              4


# Image Processing and Transforms

In [14]:
def view_img_dimensions(path : str , set):
  image = Image.open(path)
  width, height = image.size
  set.add((width , height))

In [15]:
set_dimensions = set()
for path_image in train_dfx['img_path']:
  view_img_dimensions(path_image , set_dimensions)
print(set_dimensions)


{(743, 800), (800, 742), (4000, 3000), (800, 800), (800, 681), (800, 574), (800, 757), (727, 800), (800, 668), (800, 790), (742, 800), (800, 747), (625, 800), (800, 771), (800, 710), (649, 800), (800, 600), (682, 800), (607, 800), (800, 783), (800, 722), (800, 682), (660, 800), (800, 688), (781, 800), (578, 800), (600, 800), (785, 800), (774, 800), (729, 780), (732, 800), (800, 599), (800, 657), (800, 782), (800, 721)}


In [16]:
print(f"The distinct dimensional size of images : {len(set_dimensions)}")# Different dimensions are there
def min_tuple_values(tuples_set):
    min0 = float('inf')
    min1 = float('inf')

    for a, b in tuples_set:
        min0 = min(min0, a)
        min1 = min(min1, b)

    return (min0, min1)
print("Minimum dimensions are : " , min_tuple_values(set_dimensions))

The distinct dimensional size of images : 35
Minimum dimensions are :  (578, 574)


In [17]:
train_transforms = transforms.Compose([
    transforms.Resize((224 , 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [18]:
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

## Model architectures

DataSet class

In [19]:
class LeafDataset(Dataset):
  def __init__(self , dataframe , transforms= None):
    self.df = dataframe
    self.transforms = transforms
  def __len__ (self):
    return len(self.df)
  def __getitem__(self, index):
    img_path = self.df.iloc[index]["img_path"]
    label = self.df.iloc[index]["label_encoded"]
    image = Image.open(img_path).convert("RGB")

    if self.transforms:
        image = self.transforms(image)

    return image, label

In [20]:
train_dataset = LeafDataset(train_dfx , transforms=train_transforms)
test_dataset =  LeafDataset(val_dfx , transforms=val_transform)

In [21]:
train_dataloader = DataLoader(
    train_dataset ,
    batch_size=32,
    shuffle = True,
    num_workers=4
)
test_dataloader = DataLoader(
    train_dataset ,
    batch_size=32,
    shuffle = False,
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [22]:
import torch.nn as nn
import torch.nn.functional as F

class LeafCNN(nn.Module):

    def __init__(self, num_classes=5):
        super(LeafCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Block 2
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 3
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2,2)

        # Global Average Pool
        self.gap = nn.AdaptiveAvgPool2d((1,1))

        # Classifier
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = self.gap(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
model = torchvision.models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)
model = model.to(device)

In [41]:
criterion = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters() , lr = 0.001) # Reduced learning rate for finetuning

In [42]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

## Train the models on data

In [43]:
epochs = 10

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for img, labels in train_dataloader:

        images = img.to(device)
        labels = labels.to(device)

        optim.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optim.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_dataloader)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 1.2619
Epoch 2/10, Loss: 0.8328
Epoch 3/10, Loss: 0.7040
Epoch 4/10, Loss: 0.6315
Epoch 5/10, Loss: 0.6156
Epoch 6/10, Loss: 0.5838
Epoch 7/10, Loss: 0.5445
Epoch 8/10, Loss: 0.5275
Epoch 9/10, Loss: 0.5324
Epoch 10/10, Loss: 0.5225


In [31]:
train_dfx.columns

Index(['img_path', 'label', 'label_encoded'], dtype='object')

In [44]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for img, labels in test_dataloader:

        images = img.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.834375


In [30]:
train_dfx["label"].value_counts()

,count
label,
Downy Mildew,320
Healthy Leaf,320
Bacterial Leaf Spot,320
Mosaic Disease,320
Powdery_Mildew,320


## Evaluate the models

## test the models

## Save the models

In [45]:
torch.save(model.state_dict(), "leaf_disease_model.pth")

In [59]:
import torchvision.models as models

alexnet_model = models.alexnet(pretrained=True)

num_ftrs = alexnet_model.classifier[6].in_features
alexnet_model.classifier[6] = nn.Linear(num_ftrs, 5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [60]:
vgg_model = models.vgg16(pretrained=True)

num_ftrs = vgg_model.classifier[6].in_features
vgg_model.classifier[6] = nn.Linear(num_ftrs, 5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [61]:
dense_model = models.densenet121(pretrained=True)
num_ftrs = dense_model.classifier.in_features
dense_model.classifier = nn.Linear(num_ftrs, 5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [62]:
mobile_model = models.mobilenet_v2(pretrained=True)

num_ftrs = mobile_model.classifier[1].in_features
mobile_model.classifier[1] = nn.Linear(num_ftrs, 5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [63]:
eff_model = models.efficientnet_b0(pretrained=True)

num_ftrs = eff_model.classifier[1].in_features
eff_model.classifier[1] = nn.Linear(num_ftrs, 5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [64]:
vit_model = models.vit_b_16(pretrained=True)

num_ftrs = vit_model.heads.head.in_features
vit_model.heads.head = nn.Linear(num_ftrs, 5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [67]:
for param in alexnet_model.parameters():
    param.requires_grad = False

for param in alexnet_model.classifier[6].parameters():
    param.requires_grad = True
#

for param in vgg_model.parameters():
    param.requires_grad = False

for param in vgg_model.classifier[6].parameters():
    param.requires_grad = True
#

for param in dense_model.parameters():
  param.requires_grad = False

for param in dense_model.classifier.parameters():
    param.requires_grad = True
#

for param in mobile_model.parameters():
    param.requires_grad = False

for param in mobile_model.classifier[1].parameters():
    param.requires_grad = True
#
for param in eff_model.parameters():
    param.requires_grad = False

for param in eff_model.classifier[1].parameters():
    param.requires_grad = True

  #
for param in vit_model.parameters():
    param.requires_grad = False

for param in vit_model.heads.head.parameters():
    param.requires_grad = True

In [68]:
models_dict = {
    "AlexNet": alexnet_model,
    "VGG16": vgg_model,
    "DenseNet": dense_model,
    "MobileNet": mobile_model,
    "EfficientNet": eff_model,
    "ViT": vit_model
}

In [69]:
criterion = nn.CrossEntropyLoss()

In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [73]:
epochs = 5

for name, model in models_dict.items():

    model = model.to(device)
    print(f"\nTraining {name}")

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.0003
    )

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for img, labels in train_dataloader:

            images = img.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_dataloader)

        print(f"{name} Epoch {epoch+1}/{epochs} Loss: {epoch_loss:.4f}")


Training AlexNet
AlexNet Epoch 1/5 Loss: 1.0063
AlexNet Epoch 2/5 Loss: 0.7299
AlexNet Epoch 3/5 Loss: 0.6691
AlexNet Epoch 4/5 Loss: 0.6537
AlexNet Epoch 5/5 Loss: 0.6068

Training VGG16
VGG16 Epoch 1/5 Loss: 1.2600
VGG16 Epoch 2/5 Loss: 0.9346
VGG16 Epoch 3/5 Loss: 0.8647
VGG16 Epoch 4/5 Loss: 0.8066
VGG16 Epoch 5/5 Loss: 0.7694

Training DenseNet
DenseNet Epoch 1/5 Loss: 1.4516
DenseNet Epoch 2/5 Loss: 1.1377
DenseNet Epoch 3/5 Loss: 0.9447
DenseNet Epoch 4/5 Loss: 0.8492
DenseNet Epoch 5/5 Loss: 0.7568

Training MobileNet
MobileNet Epoch 1/5 Loss: 1.3123
MobileNet Epoch 2/5 Loss: 0.9182
MobileNet Epoch 3/5 Loss: 0.7550
MobileNet Epoch 4/5 Loss: 0.6861
MobileNet Epoch 5/5 Loss: 0.6533

Training EfficientNet
EfficientNet Epoch 1/5 Loss: 1.4314
EfficientNet Epoch 2/5 Loss: 1.1430
EfficientNet Epoch 3/5 Loss: 0.9663
EfficientNet Epoch 4/5 Loss: 0.8739
EfficientNet Epoch 5/5 Loss: 0.8098

Training ViT
ViT Epoch 1/5 Loss: 1.4176
ViT Epoch 2/5 Loss: 1.0540
ViT Epoch 3/5 Loss: 0.8860
ViT 

In [74]:
def evaluate_accuracy(model, dataloader, device):

    model.eval()   # switch to evaluation mode

    correct = 0
    total = 0

    with torch.no_grad():   # disable gradient calculation

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    return accuracy

In [75]:
results = {}

for name, model in models_dict.items():

    acc = evaluate_accuracy(model, test_dataloader , device)

    results[name] = acc

print(results)

{'AlexNet': 0.798125, 'VGG16': 0.735, 'DenseNet': 0.79625, 'MobileNet': 0.808125, 'EfficientNet': 0.796875, 'ViT': 0.769375}


In [76]:
for name, model in models_dict.items():

    path = f"{name}_model.pth"

    torch.save(model.state_dict(), path)

    print(f"{name} saved successfully")

AlexNet saved successfully
VGG16 saved successfully
DenseNet saved successfully
MobileNet saved successfully
EfficientNet saved successfully
ViT saved successfully


In [ ]:
# import torch
# import torchvision.models as models
# import torch.nn as nn
# from torchvision import transforms
# from PIL import Image

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = models.resnet18(pretrained=False)
# model.fc = nn.Linear(model.fc.in_features, 5)

# model.load_state_dict(torch.load("best_leaf_model.pth", map_location=device))
# model = model.to(device)
# model.eval()

# transform = transforms.Compose([
#     transforms.Resize((224,224)),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485,0.456,0.406],
#         std=[0.229,0.224,0.225]
#     )
# ])

# img = Image.open("test_leaf.jpg").convert("RGB")
# img = transform(img).unsqueeze(0).to(device)

# with torch.no_grad():
#     output = model(img)
#     _, predicted = torch.max(output,1)

# class_names = [
#     'Bacterial Leaf Spot',
#     'Downy Mildew',
#     'Healthy Leaf',
#     'Mosaic Disease',
#     'Powdery Mildew'
# ]

# print("Prediction:", class_names[predicted.item()])

In [80]:
import sys
sys.version

'3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]'

In [82]:
import numpy , pandas , sklearn , PIL
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("albumentations:", albumentations.__version__)
print("sklearn:", sklearn.__version__)
print("Pillow:", PIL.__version__)

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
numpy: 2.0.2
pandas: 2.2.2
albumentations: 2.0.8
sklearn: 1.6.1
Pillow: 11.3.0
